In the name of God

# CA 4 - Large Language Models (Spring 2026)

- **Name: Seyed Mani Mirshabani**
- **Student ID: 810801080**

---
### Intelligent Research Assistant (LLM Agents)
This notebook contains the step-by-step implementation of an **Intelligent Research Assistant (ReAct Agent)** using LangChain and external tools. The goal is to build an agent capable of multi-hop reasoning to answer complex questions using up-to-date information and precise calculations.


#### Install the necessary packages

In [1]:
# Install the necessary packages
!pip install -q langchain langchain-google-genai python-dotenv wikipedia duckduckgo-search beautifulsoup4
!pip install -q langchain-huggingface transformers accelerate
!pip install -qU langchain_community
!pip install -U ddgs

# These packages should be enough, but if you need anything else, you can add here.

## LLM Setup (5 Points)
Use Qwen2.5 3B or 7B as a base LLM. Load the model and its tokenizer from huggingface(or any other ways to load that)


In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id = "Qwen/Qwen2.5-3B-Instruct"

# Write your code here
tokenizer = AutoTokenizer.from_pretrained(model_id)       # load the tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)           # load the model
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    return_full_text=False,
)            # create a 'pipeline' object



#---------------------------------------------------------------
hf_pipeline = HuggingFacePipeline(pipeline=pipe)
llm = ChatHuggingFace(llm=hf_pipeline)      # you will be using this object from now on! it has 'invoke()' method to work with!
print("Hugging Face LLM successfully initialized!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Hugging Face LLM successfully initialized!


## Step 1: Facing the Limitations of the Base LLM (10 Points)


In this section, we ask a multi-hop question directly to the model (without any tools) to observe its limitations, such as lack of future knowledge or math errors. Analyze the outputs.


I used the questions defined in the next step and ran it on the LLM without any tools.

In [3]:
complex_question_1 = (
    "How old would Isaac Newton be if he were alive today?"
)

complex_question_2 = (
    "Who was the winner of the Academy Award for Best Director in 2025? "
    "Find their date of birth and calculate exactly how old they will be during the 2026 FIFA World Cup. "
    "Finally, provide a 2-sentence summary of their filmmaking style."
)

print("Testing Question 1 ")
response_1 = llm.invoke(complex_question_1).content
print(f"Question: {complex_question_1}")
print(f"Base LLM Response:\n {response_1}")

print("=======================================")

print("Testing Question 2 ")
response_2 = llm.invoke(complex_question_2).content
print(f"Question: {complex_question_2}")
print(f"Base LLM Response:\n {response_2}")

Testing Question 1 


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How old would Isaac Newton be if he were alive today?
Base LLM Response:
 To determine how old Isaac Newton would be if he were alive today, we need to consider his birth and death dates.

Isaac Newton was born on December 25, 1642, and he died on March 31, 1726/27 (the Julian calendar was used then, which is why the date of his death is written as two parts).

To calculate his age today, we need to know the current year and the exact date. As of my last update in October 2023, it is 2023. 

If we assume today's date is after March 31, 2023, but before December 25, 2023, Newton would be approximately 382 years old. If today's date is after December 25, 2023, he would be 383 years old.

Since the exact date is not provided here, I will provide both possible answers:

- If today's date is before March 31, 2023, Newton would be approximately **382 years old**.
- If today's date is after March 31, 2023, Newton would be **383 years old**.
Testing Question 2 
Question: Who was the 

As we can see, the model thinks it's 2023 so it calculates the age wrong and says there is no information about the academy award!

## Step 2: Defining Tools for LangChain (30 Points)


Define six functional tools using the `@tool` decorator. These tools will be:
- web_search
- read_page
- wikipedia_search
- calculator
- summarize
- get_current_date


PAY ATTENTION: You MUST write proper and percise Docstrings for these methods, so the LLM knows exactly *when* and *how* to use them, if
you use langchain's built-in abilities to bind the tools to the chat tempalte (for example if you are talking to gemini via its API).

However, in the current assignment, we should do this binding manually due to some restrictions. (you will do this in the system prompt in a later cell.) But, it is always a good practice to write a proper Docstring for the `@tool` methods (and any other methods!).

In [4]:
complex_question_1 = (
    "How old would Isaac Newton be if he were alive today?"
)

complex_question_2 = (
    "Who was the winner of the Academy Award for Best Director in 2025? "
    "Find their date of birth and calculate exactly how old they will be during the 2026 FIFA World Cup. "
    "Finally, provide a 2-sentence summary of their filmmaking style."
)

print("Testing Question 1 ")
response_1 = llm.invoke(complex_question_1).content
print(f"Question: {complex_question_1}")
print(f"Base LLM Response:\n {response_1}")

print("=======================================")

print("Testing Question 2 ")
response_2 = llm.invoke(complex_question_2).content
print(f"Question: {complex_question_2}")
print(f"Base LLM Response:\n {response_2}")

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Testing Question 1 


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How old would Isaac Newton be if he were alive today?
Base LLM Response:
 Isaac Newton lived from January 4, 1643, to March 31, 1727, according to the Gregorian calendar. To determine how old he would be if he were alive today (which is currently the year 2023), we need to calculate the difference between these two dates.

Starting with the birth and death years:
- Birth year: 1643
- Death year: 1727

The calculation for his age in 2023 would be:

\[ \text{Age} = 2023 - 1643 = 380 \]

So, if Isaac Newton were alive today, he would be 380 years old.
Testing Question 2 
Question: Who was the winner of the Academy Award for Best Director in 2025? Find their date of birth and calculate exactly how old they will be during the 2026 FIFA World Cup. Finally, provide a 2-sentence summary of their filmmaking style.
Base LLM Response:
 I'm sorry for any confusion, but as of now (2023), there is no Academy Award ceremony scheduled for 2025 or any information about who might win the award

In [5]:
from langchain_core.tools import tool
import datetime
import requests
from bs4 import BeautifulSoup
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from simplejson import JSONDecodeError

@tool
def get_current_date():
    """
    Returns the current date as a string in the format YYYY-MM-DD.
    Use this tool whenever you need to know today's date to answer a time-sensitive question.
    """
    return datetime.datetime.now().strftime("%Y-%m-%d")

@tool
def calculator(expression: str):
    """
    Evaluates a mathematical expression and returns the result.
    Input should be a string containing a valid Python mathematical expression (e.g., '2 + 2' or '3 * 5').
    Use this tool whenever you need to perform math calculations.
    """
    return eval(expression)

@tool
def wikipedia_search(query: str):
    """
    Searches Wikipedia for a given topic and returns a summary of the page.
    Input should be a specific search query (e.g., 'Isaac Newton' or 'Academy Award for Best Director').
    Use this tool as your primary source for factual, historical, or biographical information.
    """
    api_wrapper = WikipediaAPIWrapper()
    wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
    try:
        return wiki.run(query)
    except JSONDecodeError as e:
        return f"Error: Could not decode JSON response from Wikipedia API. This might be a temporary issue or an invalid query. Details: {e}"
    except Exception as e:
        return f"An unexpected error occurred during Wikipedia search: {e}"


@tool
def web_search(query: str):
    """
    Searches the web using DuckDuckGo and returns brief snippets of information.
    Input should be a concise search query.
    Use this tool to find information about recent events, news, or topics that might not be on Wikipedia.
    """
    search = DuckDuckGoSearchRun()
    return search.run(query)

@tool
def read_page(url: str):
    """
    Extracts and returns the text content from a given web page URL.
    Input must be a valid, complete HTTP/HTTPS URL.
    Use this tool when a web search returns a specific URL and you need to read the full details of that page.
    """
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    return soup.get_text()

@tool
def summarize(text: str):
    """
    Generates a concise summary of the provided text.
    Input should be a long string of text that needs to be condensed.
    Use this tool when you extract a large amount of information (like a full web page) and need to extract just the key points.
    """
    prompt = "Summarize the following text:\n\n" + text
    answer = llm.invoke(prompt)
    return answer.content

tools_list = [
    get_current_date,
    calculator,
    wikipedia_search,
    web_search,
    read_page,
    summarize
]
print("Tools successfully Created!")

Tools successfully Created!


In [6]:
# Check each method individually on a simple input to confirm their correct functionality.

print("Testing get_current_date ")
print(get_current_date.invoke({}))

print("Testing calculator ")
print(calculator.invoke({"expression": "2026 - 1970"}))

print("Testing wikipedia_search ")
print(wikipedia_search.invoke({"query": "Isaac Newton"}))

print("Testing web_search ")
print(web_search.invoke({"query": "Who won the super bowl in 2024?"}))

print("Testing read_page ")
test_url = "https://example.com"
print(read_page.invoke({"url": test_url}))

print("Testing summarize ")
dummy_text = (
    "Artificial intelligence (AI) is intelligence demonstrated by machines, "
    "as opposed to the natural intelligence displayed by animals including humans. "
    "AI research has been defined as the field of study of intelligent agents, "
    "which refers to any system that perceives its environment and takes "
    "actions that maximize its chance of achieving its goals."
)
print(summarize.invoke({"text": dummy_text}))

Testing get_current_date 
2026-06-14
Testing calculator 
56
Testing wikipedia_search 


/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Page: Isaac Newton
Summary: Sir Isaac Newton ( ; (1643-01-04)4 January 1643 [O.S. 25 December 1642] – 31 March [O.S. 20 March] 1727) was an English polymath who was a mathematician, physicist, astronomer, alchemist, theologian, author and inventor. He was a key figure in the Scientific Revolution and the Enlightenment that followed. His book Philosophiæ Naturalis Principia Mathematica (Mathematical Principles of Natural Philosophy), first published in 1687, achieved the first great unification in physics and established classical mechanics. Newton also made seminal contributions to optics, and shares credit with the German mathematician Gottfried Wilhelm Leibniz for formulating infinitesimal calculus, although he developed calculus years before Leibniz. Newton contributed to and refined the scientific method, and his work is considered the most influential in bringing forth modern science.
In the Principia, Newton formulated the laws of motion and universal gravitation that formed the 

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Only two franchises have ever won the Super Bowl while hosting at their home stadiums: the Tampa Bay Buccaneers in Super Bowl LV and the Los Angeles Rams in Super Bowl LVI. The reigning champions are the Seattle Seahawks following their victory over the New England Patriots in Super Bowl LX. The Kansas City Chiefs won Super Bowl LVIII in 2024, defeating the San Francisco 49ers 25–22 in overtime. The game took place on February 11, 2024, at Allegiant Stadium in Las Vegas, Nevada. The Chiefs clinched another Super Bowl victory on Feb. 11, 2024, after overtaking the San Francisco 49ers in overtime with a score of 25-22 at Allegiant Stadium in Las Vegas. The 2024 Philadelphia Eagles had one of the most dominant playoff performances in recent history, and they capped it off by winning Super Bowl LIX, beating the breaks off the Kansas City... Discover who won the last 10 Super Bowls from 2016 to 2025, including scores, MVPs, and key facts about recent champions.
Testing read_page 
Example Do

#### Add Your Own Custom Tool! (5 Point)



Add a new custom tool you think can be useful for a research assistant. Then add a new *markdown* cell and explain briefly the purpose of your custom tool. Do not forget that you should use this method in the rest of the assignment.


Here we add a new tool to search papers on ArXiv

In [7]:
!pip install -U langchain-community arxiv


In [8]:
# Write your code here
import arxiv
from langchain_core.tools import tool

@tool
def arxiv_search(query: str):
    """
    Search the ArXiv database for scientific papers.
    Input: A search query string (e.g., 'Large Language Models').
    Output: A list of paper titles and brief abstracts.
    Use this tool when you need to find scientific literature or research papers.
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=3,
        sort_by=arxiv.SortCriterion.Relevance
    )

    results = []
    try:
        for paper in client.results(search):
            paper_info = (
                f"Title: {paper.title}\n"
                f"Authors: {', '.join([a.name for a in paper.authors])}\n"
                f"Summary: {paper.summary}\n"
            )
            results.append(paper_info)

        if not results:
            return "No papers found on ArXiv for this query."

        return "\n---\n".join(results)

    except Exception as e:
        return f"An error occurred while searching ArXiv: {e}"

tools_list.append(arxiv_search)
print("Testing arxiv_search ")
arxiv_search.invoke({"query": "Large Language Models"})

Testing arxiv_search 


"Title: Enhancing Human-Like Responses in Large Language Models\nAuthors: Ethem Yağız Çalık, Talha Rüzgar Akkuş\nSummary: This paper explores the advancements in making large language models (LLMs) more human-like. We focus on techniques that enhance natural language understanding, conversational coherence, and emotional intelligence in AI systems. The study evaluates various approaches, including fine-tuning with diverse datasets, incorporating psychological principles, and designing models that better mimic human reasoning patterns. Our findings demonstrate that these enhancements not only improve user interactions but also open new possibilities for AI applications across different domains. Future work will address the ethical implications and potential biases introduced by these human-like attributes.\n\n---\nTitle: Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality\nAuthors: Yiming Ai, Zhiwei He, Ziyin Zhang, Wenhong Zhu, Hongkun Hao, 

## Step 3: Single-Turn Action & Parser (5 Points)


Create a custom parser using Regular Expressions (Regex) to extract the action requests from the LLM's output and then test the function.

- Note: Regex is a simple way. If you want do something else to parse agent's outputs, you are free to do so as long as the code runs correctly.
- Note2: You can instruct the agent to generate its output in a different format. If you are using a different output format, you can modify the samples in the following cell. (you will do this in the system prompt in a later cell.)

In [9]:
import re

def parse_action(text: str):
    """
    A fortified parser designed to handle the syntax quirks of local open-weights models.
    """
    pattern = r"Action:\s*(.*?)\s*\[(.*?)\]"
    match = re.search(pattern, text, flags=re.DOTALL)

    if match:
        tool_name = match.group(1).strip()
        tool_input = match.group(2).strip()
        return tool_name, tool_input

    return None, None

# --- Testing the Fortified Version with typical "small model" flaws ---
print("Test 1 (Hyphens & Spaces):", parse_action("Action: web-search  [\n2025 oscar winner]"))
print("Test 2 (Trailing Garbage):", parse_action("Action: calculator[2026 - 1970]\nObservation: I shouldn't write this."))

Test 1 (Hyphens & Spaces): ('web-search', '2025 oscar winner')
Test 2 (Trailing Garbage): ('calculator', '2026 - 1970')


## Step 4: ReAct Loop and Memory Management (30 Points)



Build the intelligent loop that repeats the process of thinking, deciding, applying a tool, and receiving an observation.

NOTE: Don't forget to log intermediary steps and outputs! It is important.

In [20]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import torch

def run_react_agent(question: str, max_iterations: int = 7):

    tools_map = {tool.name: tool for tool in tools_list}
    tools_description = "\n".join([f"- {tool.name}: {tool.description}" for tool in tools_list])

    system_prompt = f"""You are a helpful reasoning assistant. You have access to the following tools:

{tools_description}

You must solve the user's problem by thinking step-by-step. You must always check for
information you may not know or you may think could have been updated such as date,
historical events after your training, etc. You must ALWAYS use the following exact format:

Question: the input question you must answer
Thought: you should always think about what to do next
Action: tool_name[tool_input] (If the tool requires no input, you MUST use empty brackets, e.g., tool_name[])
Observation: the result of the action

... (this Thought/Action/Observation can repeat N times) ...

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!
"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Question: {question}")
    ]

    for i in range(max_iterations):
        print(f"\n{'-'*15} Cycle {i+1} {'-'*15}")

        # Write your code here
        # Do not forget to process and polish the outputs, and then print them!
        # The printed outputs must be human-readable, without extra tokens(e.g. |im_start| or |im_end| or ...)
        # In each iteration, you should pass the `messages` to the model, receive a generated response. parse the output,
        # do what is necessary (for example call a tool!) and append something to `messages` for the next iteration.
        response = llm.bind(stop=["\nObservation:", "Observation:"]).invoke(messages)
        # response = llm.invoke(messages, stop=["Observation:", "\nObservation:"])
        llm_output = response.content
        llm_output = llm_output.replace("<|im_end|>", "").strip()

        llm_output = llm_output.split("Observation:")[0].strip()

        print(f"Agent Output:\n{llm_output}\n")

        tool_name, tool_input = parse_action(llm_output)
        if tool_name is None:
            if "Final Answer:" in llm_output:
                print("Agent found the final answer!")
                break
            else:
                print("Format error")
                messages.append(AIMessage(content=llm_output))
                messages.append(HumanMessage(content="Observation: Format error please use the 'Action: tool_name[tool_input]' format or provide a 'Final Answer:'."))
                continue
        print(f"Parsed Tool: {tool_name} \nParsed Input: {tool_input}")

        if tool_name not in tools_map:
            observation = f"Error: Tool '{tool_name}' not found, please use one of the available tools."
        else:
            selected_tool = tools_map[tool_name]
            observation = selected_tool.invoke(tool_input)

        print(f"Observation: {observation}")

        messages.append(AIMessage(content=llm_output))
        messages.append(HumanMessage(content=f"Observation: {observation}"))

    return messages

In [21]:
# Run the agent with your variable setup
final_messages = run_react_agent("How old would Isaac Newton be if he were alive today?")

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------- Cycle 1 ---------------
Agent Output:
Thought: To determine how old Isaac Newton would be if he were alive today, I need to first find his birth year and then calculate the difference between that year and the current year. I'll use Wikipedia to find his birth year.
Action: wikipedia_search[Isaac Newton]

Parsed Tool: wikipedia_search 
Parsed Input: Isaac Newton


/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: Page: Isaac Newton
Summary: Sir Isaac Newton ( ; (1643-01-04)4 January 1643 [O.S. 25 December 1642] – 31 March [O.S. 20 March] 1727) was an English polymath who was a mathematician, physicist, astronomer, alchemist, theologian, author and inventor. He was a key figure in the Scientific Revolution and the Enlightenment that followed. His book Philosophiæ Naturalis Principia Mathematica (Mathematical Principles of Natural Philosophy), first published in 1687, achieved the first great unification in physics and established classical mechanics. Newton also made seminal contributions to optics, and shares credit with the German mathematician Gottfried Wilhelm Leibniz for formulating infinitesimal calculus, although he developed calculus years before Leibniz. Newton contributed to and refined the scientific method, and his work is considered the most influential in bringing forth modern science.
In the Principia, Newton formulated the laws of motion and universal gravitation tha

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: I found Isaac Newton's birth year in the Wikipedia summary, but I don't have the current date yet. I will use the `get_current_date` tool to retrieve today's date and then subtract Newton's birth year from it to find out how old he would be today.
Action: get_current_date[]

Parsed Tool: get_current_date 
Parsed Input: 
Observation: 2026-06-14

--------------- Cycle 3 ---------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: Now that I have the current date, I can subtract Newton's birth year from it to find out how old he would be today. Let's calculate it.
Action: calculator['2026 - 1643']

Parsed Tool: calculator 
Parsed Input: '2026 - 1643'
Observation: 2026 - 1643

--------------- Cycle 4 ---------------
Agent Output:
Thought: The calculator returned the result directly, so I can now present the final answer.
Final Answer: If Isaac Newton were alive today, he would be 383 years old.

Agent found the final answer!


In [22]:
complex_question = (
    "Who was the winner of the Academy Award for Best Director in 2025? "
    "Find their date of birth and calculate exactly how old they will be during the 2026 FIFA World Cup. "
    "Finally, provide a 2-sentence summary of their filmmaking style."
)
final_messages = run_react_agent(complex_question)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------- Cycle 1 ---------------
Agent Output:
Thought: The question asks for information about the winner of the Academy Award for Best Director in 2025, which is not possible as the event hasn't happened yet. However, we can make an assumption that the question intends to ask about a past winner and proceed accordingly. To find this information, I will first perform a web search.
Action: web_search["winner of the Academy Award for Best Director in 2025"]

Parsed Tool: web_search 
Parsed Input: "winner of the Academy Award for Best Director in 2025"


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: 1 week ago - Ford (1940–1941), Joseph L. Mankiewicz (1949–1950), and Alejandro González Iñárritu (2014–2015) are the only directors to have won the award in two consecutive years. Francis Ford Coppola is the only director to be nominated for each film of a trilogy, The Godfather trilogy, winning for the second film. Four directing teams have been nominated together (a total of five times, winning on three occasions): Robert Wise and Jerome Robbins for West Side Story (1961, winners); Warren Beatty and Buck Henry for Heaven Can Wait (1978); Joel and Ethan Coen for No Country for Old Men (2007, winners) and True Grit (2010); and Daniel Kwan and Daniel Scheinert for Everything Everywhere All at Once (2022, winners). 1 day ago - The Academy Scientific and Technical Awards were presented by host Diego Luna on April 29, 2025, in a ceremony at the Academy Museum of Motion Pictures in Los Angeles. Anora won a leading five awards, including Best Picture. Other winners included The 

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Agent Output:
Thought: I have gathered all the required information to answer the question. First, I will calculate Paul Thomas Anderson's age in 2026. Then, I will provide the summary of his filmmaking style. Finally, I will answer the original question.
Action: calculator["2026 - 1970"]

Parsed Tool: calculator 
Parsed Input: "2026 - 1970"
Observation: 2026 - 1970

--------------- Cycle 5 ---------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: The calculation shows that Paul Thomas Anderson is 56 years old in 2026. Now I can answer the original question. I will provide a 2-sentence summary of his filmmaking style based on the information found.
Action: summarize["Paul Thomas Anderson (born June 26, 1970), also known by his initials PTA, is an American filmmaker. Often described as one of the preeminent filmmakers of his generation, he is the recipient of three Academy Awards, three Golden Globe Awards and four BAFTA Awards, as well as a nomination for ..."]

Parsed Tool: summarize 
Parsed Input: "Paul Thomas Anderson (born June 26, 1970), also known by his initials PTA, is an American filmmaker. Often described as one of the preeminent filmmakers of his generation, he is the recipient of three Academy Awards, three Golden Globe Awards and four BAFTA Awards, as well as a nomination for ..."


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: Paul Thomas Anderson, born on June 26, 1970, is an American filmmaker widely regarded as one of the most significant in his generation. He has received multiple prestigious accolades including three Academy Awards, three Golden Globe Awards, and four BAFTA Awards. Additionally, he has been nominated for another award.

--------------- Cycle 6 ---------------
Agent Output:
Thought: I now know the final answer.
Final Answer: Paul Thomas Anderson, born on June 26, 1970, is an American filmmaker who was 56 years old in 2026. As per the information, he will be old enough to attend the 2026 FIFA World Cup. A 2-sentence summary of his filmmaking style would be: Paul Thomas Anderson is an American filmmaker widely regarded as one of the most significant in his generation. He has received multiple prestigious accolades including three Academy Awards, three Golden Globe Awards, and four BAFTA Awards.

Agent found the final answer!


  Initially the agent would hallucinate the rest of the conversation and continue based on imaginary tool outputs, after some tinkering and failing I had to manually remove the haucination part so the tool use would go as planned. Once the tool use was fixed we can see that the answer to the first, simpler question was successful, the modle correctly used the tools to extract the bird date and the current date but it failed when using the calculator (formatting issues) however it managed to do it internally in the next cycle so the final answer was calculated correctly!

For te second questions however after multiple try and errors didn't manage to produce the correct output, I suspect this is due to the small size of the model as the problem seems to be the model's ability to handle inputs as it really didn't improve with different system prompts, for the current run we can see the following:

The model managed to understand the date correctly (and didn't think it's still 2023!), it managed to extract the birthdate of the actor using the tools but it didn't manage to find the correct actor. As we can see the search results mention two oscar winning actors Sean Baker (winning the 2025 Oscar for Anora) and Paul Thomas Anderson (winning a 2026 Oscar). The model here fails to stick the original prompt and instead of focusing on the 2025 winner as mentioned in the prompt it shifts its focus on the 2026 winner and thus providing the wrong answer! Another point of failure is that in cycle 5 the model calls the summerize tool however it doesn't provide the correct text to it! Instead of giving it the filmmaking style text to summerize, it gives it a text about the biography of the actor and as a result the summery becomes incorrect!

So in conclusion the ReAct enabled the model to answer multihop questions and answer some questions that it was previously unable to answer but there are still limits to how far it can improve the model and it mainly comes down to the size and knowledge of the model itsefl!

## Step 5: Robustness Testing and Tool Error Management (Reflection) (5 Points)




Test the system with a deceptive question (where the data does not exist on Wikipedia) to evaluate its error-handling and redirection logic.

In [23]:
tricky_question = "What is the birth date of the King of the fictional country 'Zimbagweland' on Wikipedia? If the page is not found, analyze the reason why."

print("Starting robustness test...\n")
run_react_agent(tricky_question, max_iterations=4)


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting robustness test...


--------------- Cycle 1 ---------------
Agent Output:
Thought: To find the birth date of the King of the fictional country 'Zimbagweland', we first need to locate the relevant Wikipedia page. If it's not found, we will attempt to search for it on the web.
Action: wikipedia_search['King of Zimbagweland']

Parsed Tool: wikipedia_search 
Parsed Input: 'King of Zimbagweland'


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: No good Wikipedia Search Result was found

--------------- Cycle 2 ---------------
Agent Output:
Thought: The Wikipedia search did not return any results, so let's try searching the web for more information on the birth date of the King of Zimbagweland.
Action: web_search['birth date of King of Zimbagweland']

Parsed Tool: web_search 
Parsed Input: 'birth date of King of Zimbagweland'


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: Archaeological records date archaic human settlement of present-day Zimbabwe to at least 500,000 years ago. [28] Zimbabwe's earliest known inhabitants were most likely the San people, who left behind a legacy of arrowheads and cave paintings. King Shaka was born in the lunar month of uNtulikazi (July) in 1787, in Mthonjaneni, KwaZulu-Natal Province, South Africa. The son of the Zulu King Senzangakhona kaJama, he was spurned as an illegitimate son. King Shaka was born in the lunar month of uNtulikazi (July) in 1787, in Mthonjaneni, KwaZulu-Natal Province, South Africa. The son of the Zulu King Senzankakhona kaJama, he was spurned as an illegitimate son. Jul 27, 2025 · He was born Sigidi kaSenzangakhona near present-day Melmoth, KwaZulu-Natal Province. Due to persecution due to his illegitimacy, he spent his childhood in his mother's settlements, where he was initiated into an ibutho lempi (fighting unit). Tshaka (also known as Chaka, Shaka) was born an unwanted son of a min

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: No good Wikipedia Search Result was found

--------------- Cycle 4 ---------------
Agent Output:
Thought: Since we still cannot find the birth date of the King of the fictional country 'Zimbagweland' on either Wikipedia or the web, I am going to conclude that the information does not exist. 
Final Answer: Unfortunately, based on the available information, there is no birth date for the King of the fictional country 'Zimbagweland'.

Agent found the final answer!


[SystemMessage(content="You are a helpful reasoning assistant. You have access to the following tools:\n\n- get_current_date: Returns the current date as a string in the format YYYY-MM-DD.\nUse this tool whenever you need to know today's date to answer a time-sensitive question.\n- calculator: Evaluates a mathematical expression and returns the result.\nInput should be a string containing a valid Python mathematical expression (e.g., '2 + 2' or '3 * 5').\nUse this tool whenever you need to perform math calculations.\n- wikipedia_search: Searches Wikipedia for a given topic and returns a summary of the page.\nInput should be a specific search query (e.g., 'Isaac Newton' or 'Academy Award for Best Director').\nUse this tool as your primary source for factual, historical, or biographical information.\n- web_search: Searches the web using DuckDuckGo and returns brief snippets of information.\nInput should be a concise search query.\nUse this tool to find information about recent events, ne

The most important success of this section is that the model didn't hallucinate! It did not make up information about Zimbagweland which was something we expected as these smaller models are highly prone to hallucination. It also successfully executed the right steps for example it first used wikipedia and when it didn't get the results it used the websearch. It was however not without mistakes, despite knowing the wikipeida didn't give any results, the model attempted using it again in cycle 3. The prompt said If the page is not found, analyze the reason why. but despite the model currectly finding that the information indeed doesn't exist, it igonred this section and didn't address it!

Overall the performance was acceptable, it uses some extra tools and didn't quite answer everything in the prompt but I think the most important part was that it didn't hallucinate!

## Step 6: Experiment (15 Points)



Design 5 examples and analyze the answer of the agent for each example.

#### Example one:

In [24]:
# Write your code here
question_1 = "Search Arxiv for a paper about 'Low-Rank Adaptation (LoRA) for Large Language Models' and summarize its main findings."
print("Starting Example 1:")
run_react_agent(question_1)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Example 1:

--------------- Cycle 1 ---------------
Agent Output:
Thought: First, I need to search the ArXiv database for papers related to 'Low-Rank Adaptation (LoRA) for Large Language Models'. After obtaining the results, I will then select one of those papers and summarize its main findings.
Action: arxiv_search["Low-Rank Adaptation (LoRA) for Large Language Models"]

Parsed Tool: arxiv_search 
Parsed Input: "Low-Rank Adaptation (LoRA) for Large Language Models"


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: Title: LoTR: Low Tensor Rank Weight Adaptation
Authors: Daniel Bershatsky, Daria Cherniuk, Talgat Daulbaev, Aleksandr Mikhalev, Ivan Oseledets
Summary: In this paper we generalize and extend an idea of low-rank adaptation (LoRA) of large language models (LLMs) based on Transformer architecture. Widely used LoRA-like methods of fine-tuning LLMs are based on matrix factorization of gradient update. We introduce LoTR, a novel approach for parameter-efficient fine-tuning of LLMs which represents a gradient update to parameters in a form of tensor decomposition. Low-rank adapter for each layer is constructed as a product of three matrices, and tensor structure arises from sharing left and right multipliers of this product among layers. Simultaneous compression of a sequence of layers with low-rank tensor representation allows LoTR to archive even better parameter efficiency then LoRA especially for deep models. Moreover, the core tensor does not depend on original weight dimens

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: I realize there was a mistake in my previous thought. The summary for the paper "LoTR: Low Tensor Rank Weight Adaptation" was not properly extracted from the provided information. I need to re-examine the summary provided and extract the main findings.
Action: extract_main_findings_from_summary[LoTR's summary]
Thought: Having the main findings of the paper "LoTR: Low Tensor Rank Weight Adaptation", I can now generate the final answer.
Action: Thought: I now know the final answer
Final Answer: The main findings of the paper "LoTR: Low Tensor Rank Weight Adaptation" include the introduction of a novel approach for parameter-efficient fine-tuning of large language models (LLMs) called LoTR. LoTR represents a gradient update to parameters in a form of tensor decomposition, where a low-rank adapter for each layer is constructed as a product of three matrices. This tensor structure arises from sharing left and right multipliers of the product among layers, allowing for

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: I seem to have made an error in my previous steps. The tool 'extract_main_findings_from_summary' was not correctly named. I should directly extract the main findings from the provided summary instead of creating a fictional tool.
Action: extract_main_findings_from_summary[LoTR's summary]
Thought: Now I have the main findings of the paper "LoTR: Low Tensor Rank Weight Adaptation". I can now generate the final answer.
Action: Thought: I now know the final answer
Final Answer: The main findings of the paper "LoTR: Low Tensor Rank Weight Adaptation" include the introduction of a novel approach for parameter-efficient fine-tuning of large language models (LLMs) called LoTR. LoTR represents a gradient update to parameters in a form of tensor decomposition, where a low-rank adapter for each layer is constructed as a product of three matrices. This tensor structure arises from sharing left and right multipliers of the product among layers, allowing for even better parame

[SystemMessage(content="You are a helpful reasoning assistant. You have access to the following tools:\n\n- get_current_date: Returns the current date as a string in the format YYYY-MM-DD.\nUse this tool whenever you need to know today's date to answer a time-sensitive question.\n- calculator: Evaluates a mathematical expression and returns the result.\nInput should be a string containing a valid Python mathematical expression (e.g., '2 + 2' or '3 * 5').\nUse this tool whenever you need to perform math calculations.\n- wikipedia_search: Searches Wikipedia for a given topic and returns a summary of the page.\nInput should be a specific search query (e.g., 'Isaac Newton' or 'Academy Award for Best Director').\nUse this tool as your primary source for factual, historical, or biographical information.\n- web_search: Searches the web using DuckDuckGo and returns brief snippets of information.\nInput should be a concise search query.\nUse this tool to find information about recent events, ne

The goal for this example was to use the costume tool and see if the model can handle it or not. It successfully called the arxiv_search tool in cycle and in cucle 2 it unfortunately failed as the model passed the wrong text to the summerize tool and when the tool failed in cycles 3 and 4 the mode began halucinating and tried to use tools that didn't exist!

But by cycle 5 it managed to get over the broken tool chain and read the ArXiv paper itself and manually extract the information from it so at the end it managed to produce a good answer!

#### Example two:

In [29]:
# Write your code here
question_2 = "Exactly how many years has it been since the release of the first iPhone?"
print("Starting Example 2:")
run_react_agent(question_2, max_iterations=6)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Example 2:

--------------- Cycle 1 ---------------
Agent Output:
Thought: To determine the number of years since the release of the first iPhone, I will need to get the release year of the first iPhone and compare it with the current year. I can find this information through a web search.
Action: web_search["first iPhone release year"]

Parsed Tool: web_search 
Parsed Input: "first iPhone release year"


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: 2 days ago - The iPhone is a smartphone developed ... and speculation, it was officially announced on January 9, 2007, and was released in the United States on June 29, 2007.... 2 days ago - The first-generation iPhone was announced by then–Apple CEO and co-founder Steve Jobs on January 9, 2007, at Macworld 2007, and launched later that year. Since then, Apple has annually released new iPhone models and iOS versions; the latest models are the iPhone 17, the higher-end iPhone ... 1 week ago - The history of the iPhone by Apple Inc. began in the early 2000s. The first iPhone was unveiled at Macworld 2007 and released later that year. By the end of 2009, iPhone models had been released in all major markets. In the early 2000s, the success of Apple's iPod, which dominated the digital ... May 4, 2026 - The iPhone (also called the first-generation iPhone, iPhone (original), iPhone 2G, and iPhone 1) was the first smartphone made by Apple, Inc. It was announced on January 9, 2007,

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: Based on the information obtained from the web search, the first iPhone was released in 2007. Now, I will calculate the difference between the current year and 2007 to determine how many years have passed.
Action: get_current_date[]

Parsed Tool: get_current_date 
Parsed Input: 
Observation: 2026-06-14

--------------- Cycle 3 ---------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: With the current date known, I can calculate the difference between the current year and the year the first iPhone was released to determine the number of years that have passed.
Action: calculator["(2026 - 2007)"]

Parsed Tool: calculator 
Parsed Input: "(2026 - 2007)"
Observation: (2026 - 2007)

--------------- Cycle 4 ---------------
Agent Output:
Thought: The calculation shows that 19 years have passed since the release of the first iPhone.
Final Answer: 19 years

Agent found the final answer!


[SystemMessage(content="You are a helpful reasoning assistant. You have access to the following tools:\n\n- get_current_date: Returns the current date as a string in the format YYYY-MM-DD.\nUse this tool whenever you need to know today's date to answer a time-sensitive question.\n- calculator: Evaluates a mathematical expression and returns the result.\nInput should be a string containing a valid Python mathematical expression (e.g., '2 + 2' or '3 * 5').\nUse this tool whenever you need to perform math calculations.\n- wikipedia_search: Searches Wikipedia for a given topic and returns a summary of the page.\nInput should be a specific search query (e.g., 'Isaac Newton' or 'Academy Award for Best Director').\nUse this tool as your primary source for factual, historical, or biographical information.\n- web_search: Searches the web using DuckDuckGo and returns brief snippets of information.\nInput should be a concise search query.\nUse this tool to find information about recent events, ne

The first attempt at running this example failed because the model was refusing to output the correct format for the tools so I ran it again and this time it went smoothly. As we can see there are no syntax issues but for some reason the no matter what the model refuses to use the correct format for using the calculator tool and it keeps puutting extra"" around it, since the model managed to successfully calculate the result on its own at the end it managed to produce the correct results by succesfully getting the release date and current date!

#### Example three:

In [26]:
# Write your code here
question_3 = "Read the Wikipedia page for 'Geoffrey Hinton' (use the URL: https://en.wikipedia.org/wiki/Geoffrey_Hinton) and tell me what university he received his PhD from."
print("Starting Example 3:")
run_react_agent(question_3, max_iterations=4)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Example 3:

--------------- Cycle 1 ---------------
Agent Output:
Thought: First, I need to fetch the Wikipedia page for Geoffrey Hinton to extract the relevant information.
Action: wikipedia_search["Geoffrey Hinton"]

Parsed Tool: wikipedia_search 
Parsed Input: "Geoffrey Hinton"


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: Page: Geoffrey Hinton
Summary: Geoffrey Everest Hinton (born 6 December 1947) is a British-Canadian computer scientist, cognitive scientist, cognitive psychologist and Nobel Prize laureate known for his work on artificial neural networks, which earned him the title "the Godfather of AI". He is University Professor Emeritus at the University of Toronto.
From 2013 to 2023, he divided his time working for Google Brain and the University of Toronto before publicly announcing his departure from Google in May 2023, citing concerns about the many risks of artificial intelligence (AI) technology. In 2017, he co-founded and became the chief scientific advisor of the Vector Institute in Toronto.
With David Rumelhart and Ronald J. Williams, Hinton was co-author of a highly cited paper published in 1986 that popularised the backpropagation algorithm for training multi-layer neural networks, although they were not the first to propose the approach. Hinton is viewed as a leading figure 

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: AI luminary and computer scientist Geoffrey Hinton, PhD,’s 2016 prediction that AI would be able to do all of the things radiologists can do within five years was highly controversial in the... A Prize for a Prize. In the early 1980s, Terrence Sejnowski and Geoffrey Hinton collaborated on brain research they thought might win them the Nobel Prize in Physiology or Medicine. In May, Geoffrey Hinton, PhD, an AI groundbreaker who helped produce technologies that led to ChatGPT, announced that he had left Google and began raising alarms about the... Author: Geoffrey. Hinton, PhD, Google. Brain Teamand.Canada (geoffrey. .hinton@gmail.com). Opinion. jama.com (Reprinted) JAMA September 18, 2018 Volume 320,Number 11 1101. Langlotz launched into this part of his presentation by quoting the psychologist, computer scientist and AI pioneer Geoffrey Hinton, PhD, who told a New Yorker writer in 2016: “[I]f you work as...

--------------- Cycle 3 ---------------
Agent Output:
Thought: Fro

[SystemMessage(content="You are a helpful reasoning assistant. You have access to the following tools:\n\n- get_current_date: Returns the current date as a string in the format YYYY-MM-DD.\nUse this tool whenever you need to know today's date to answer a time-sensitive question.\n- calculator: Evaluates a mathematical expression and returns the result.\nInput should be a string containing a valid Python mathematical expression (e.g., '2 + 2' or '3 * 5').\nUse this tool whenever you need to perform math calculations.\n- wikipedia_search: Searches Wikipedia for a given topic and returns a summary of the page.\nInput should be a specific search query (e.g., 'Isaac Newton' or 'Academy Award for Best Director').\nUse this tool as your primary source for factual, historical, or biographical information.\n- web_search: Searches the web using DuckDuckGo and returns brief snippets of information.\nInput should be a concise search query.\nUse this tool to find information about recent events, ne

The goal of this exmple was to see how well can the model handle extracting key information from a large pool. As we can see the model has failed and there are several issues/reasons for it. First the model completely neglected the user prompt. If we pay attention we can see that there clearly is a link to the wikipedia search in the prompt that the model had to use but instead it got the tools mixed up and attempted to use the wikipedia search tool instead of just searching for the link!

The wikipedia search summary didn't provide the exact information so the model fell back on websearch but this too didn't find the correct information, instead of finding the university in which Hinton got his PHD from the model saw the University of Toronto (where he is apparently a professor of) and confused this information thus it reached the wrong final answer. Here we can see that a small model with 3B parameters have a hard time tretrieving exact information and sticking to the original user prompt.

#### Example four:

In [30]:
# Write your code here
question_4 = "Who is the CEO of Apple, and what is the difference between Apple the tech company and Apple Records?"
print("Starting Example 4:")
run_react_agent(question_4, max_iterations=6)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Example 4:

--------------- Cycle 1 ---------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: To answer the question, I will first find out who the CEO of Apple is. For this, I can use the Wikipedia_search tool.
Action: wikipedia_search[CEO of Apple]

Parsed Tool: wikipedia_search 
Parsed Input: CEO of Apple
Observation: Error: Could not decode JSON response from Wikipedia API. This might be a temporary issue or an invalid query. Details: Expecting value: line 1 column 1 (char 0)

--------------- Cycle 2 ---------------
Agent Output:
Thought: The Wikipedia_search tool did not work properly due to an error. I will try to find this information through a web_search instead.
Action: web_search[CEO of Apple]

Parsed Tool: web_search 
Parsed Input: CEO of Apple


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: From the search results, it appears there has been a long-standing legal dispute between Apple Corps (formerly known as Apple Records) and Apple Inc. (formerly known as Apple Computer). The main difference between these entities seems to be the nature of their businesses - Apple Corps is a music company while Apple Inc. is a technology company. However, they share the same name.
Action: summarize[Observation]

Parsed Tool: summarize 
Parsed Input: Observation


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Observation: I'm sorry, but you haven't provided any text for me to summarize. Could you please provide the text you'd like summarized?

--------------- Cycle 5 ---------------
Agent Output:
Thought: It seems the Observation step didn't provide any text for summarization. Let me rephrase my plan. I will use the previous Observation to directly answer the question without summarization.
Action: Thought: I now know the final answer
Final Answer: The current CEO of Apple is Tim Cook. There has been a long-standing legal dispute between Apple Corps (formerly known as Apple Records) and Apple Inc. (formerly known as Apple Computer). Apple Corps is a music company owned by The Beatles, while Apple Inc. is a technology company. They share the same name, which has caused confusion and legal battles over trademark infringement.

Agent found the final answer!


[SystemMessage(content="You are a helpful reasoning assistant. You have access to the following tools:\n\n- get_current_date: Returns the current date as a string in the format YYYY-MM-DD.\nUse this tool whenever you need to know today's date to answer a time-sensitive question.\n- calculator: Evaluates a mathematical expression and returns the result.\nInput should be a string containing a valid Python mathematical expression (e.g., '2 + 2' or '3 * 5').\nUse this tool whenever you need to perform math calculations.\n- wikipedia_search: Searches Wikipedia for a given topic and returns a summary of the page.\nInput should be a specific search query (e.g., 'Isaac Newton' or 'Academy Award for Best Director').\nUse this tool as your primary source for factual, historical, or biographical information.\n- web_search: Searches the web using DuckDuckGo and returns brief snippets of information.\nInput should be a concise search query.\nUse this tool to find information about recent events, ne

For this example we can see that in cycle 1 the wikipedia tool fails because of a json error but the model correctly navigates around it and tries a websearch instead. the cycles 2 and 3 go well but we see an intersting mistake in cycle 4 where the model tries to summerize by using summerize[Observation] thinking there is some sort of variable! This is probalby due to the limitation of this small model.

But it manages to do the summerization on its own and arrive at the final correct answer. It's worth mentioning that the model failed at the inital attempt but succeded when I re-ran the code which suggests its behaiviour isn't very stable.

#### Example five:

In [31]:
# Write your code here
question_5 = "If I travel at the speed of light from Earth to Mars when they are closest to each other, how many minutes will it take?"
print("Starting Example 5:")
run_react_agent(question_5, max_iterations=7)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Example 5:

--------------- Cycle 1 ---------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: The speed of light is approximately 299,792 kilometers per second. However, we cannot actually reach the speed of light because it would require infinite energy. The distance between Earth and Mars varies but when they are closest, the average distance is around 54.6 million kilometers. We can calculate the time it takes to travel this distance at the speed of light using the formula time = distance / speed. To convert the time into minutes, we will divide the result by 60. I'll use the calculator tool to perform the calculation.
Action: calculator["54600000 / 299792"]

Parsed Tool: calculator 
Parsed Input: "54600000 / 299792"
Observation: 54600000 / 299792

--------------- Cycle 2 ---------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: The result from the previous calculation shows the time in minutes. I'll round this number to the nearest whole number since time is typically measured in whole minutes.
Action: round["observation_from_previous_step"]

Parsed Tool: round 
Parsed Input: "observation_from_previous_step"
Observation: Error: Tool 'round' not found, please use one of the available tools.

--------------- Cycle 4 ---------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Agent Output:
Thought: It seems that the 'round' function is not available. Instead, I can simply use the integer part of the result from the previous calculation since we're dealing with time. I'll use the floor function to achieve this.
Action: calculator["result_of_previous_calculation // 1"]

Parsed Tool: calculator 
Parsed Input: "result_of_previous_calculation // 1"
Observation: result_of_previous_calculation // 1

--------------- Cycle 5 ---------------
Agent Output:
Thought: The result indicates the time it takes to travel from Earth to Mars at the speed of light. Now, I need to present this as a human-readable answer in minutes.
Final Answer: At the speed of light, it would take approximately 845 minutes to travel from Earth to Mars when they are closest to each other.

Agent found the final answer!


[SystemMessage(content="You are a helpful reasoning assistant. You have access to the following tools:\n\n- get_current_date: Returns the current date as a string in the format YYYY-MM-DD.\nUse this tool whenever you need to know today's date to answer a time-sensitive question.\n- calculator: Evaluates a mathematical expression and returns the result.\nInput should be a string containing a valid Python mathematical expression (e.g., '2 + 2' or '3 * 5').\nUse this tool whenever you need to perform math calculations.\n- wikipedia_search: Searches Wikipedia for a given topic and returns a summary of the page.\nInput should be a specific search query (e.g., 'Isaac Newton' or 'Academy Award for Best Director').\nUse this tool as your primary source for factual, historical, or biographical information.\n- web_search: Searches the web using DuckDuckGo and returns brief snippets of information.\nInput should be a concise search query.\nUse this tool to find information about recent events, ne

We here see that the model entirely skipped searching the web for the parameters and used its own memory! After searching myself it seemed like the numbers were suprisingly correct!

However the mode faled in steps 2-4 where it tried to pass expressions that weren't mathematically correct! it was trying to use some variables and explanation which shows it's not fully understanding how math in python works. It also hallucinated a tool in cycle 3.

After failing to use the calculator, it fell back on doing the calculation on its own and all I can say is these small models are quite bad at doing math :)

# **Note on LLM usage:**

I used Gemini 3.1 Pro (from GapGPT) to help me with this assignment. I only used to check my answers, teach the syntax and a few conceptual questions here and there. I made a GapGPT project and set the following as the main prompt (It is probably the system prompt but I'm not sure how GapGPT works) so the LLM would only help me with learning and not just giving the answer: (This prompt was also generate by Gemini as an improved version of my own prompt :) )

The links to the conversations: 

- https://gapgpt.app/share/13xkpqwy-9low-g4ao-b0ds-g621fac7xg1wvy7
- https://gapgpt.app/share/13xkpqwy-9lpm-wtcx-q2o6-kggu6wagjw2crkp

[Role Description]
You are an expert AI/ML Professor and Senior Machine Learning Engineer. Your goal is to help the user complete their university LLM course assignments and learn the underlying concepts. The user already knows Python, but you must actively teach them frameworks like PyTorch, Hugging Face `transformers`, LangChain, and the OpenAI API.

[Core Objective]
By default, you are a strictly educational mentor. Do NOT provide direct, full solutions to assignment questions. Instead, guide the user to the answer by explaining concepts, providing documentation links, offering small conceptual code snippets, and asking leading questions. 

CRITICAL BEHAVIORS & PACING:
- Break complex LLM/PyTorch concepts down step-by-step. Never rush through multiple phases of an assignment at once. Provide a hint or a small piece of the puzzle, then PAUSE and ask the user to implement it or share their thoughts before moving on.
- API Accuracy: When teaching LangChain, Hugging Face, or OpenAI APIs, ensure your code uses the most modern, up-to-date syntax. Do not hallucinate deprecated API endpoints.

STATE MANAGEMENT & CONTEXT:
Assignments will often span multiple sessions. 
- At the beginning of a session, check if the user has provided an `Assignment_State.md` file or text. If so, read it to understand exactly where the last session left off.
- When the user indicates they are done for the day, generate a brief State Tracking Document formatted as markdown (e.g., `Assignment_State.md`). This document must include: 1) The overarching goal of the assignment, 2) What has been successfully implemented so far, 3) The exact bugs or next steps currently pending, and 4) Any specific libraries currently imported.

FORMATTING & STANDARDS:
- Format your responses with a Jupyter Notebook workflow in mind: use clear markdown for explanations, followed by distinct `python` code blocks for implementation.
- You MUST use $...$ or $$...$$ delimiters for ALL math expressions, formulas, and technical calculations (e.g., $$loss = -\frac{1}{N} \sum_{i=1}^{N} y_i \log(\hat{y}_i)$$).

ANTI-PROMPTS:
- Do NOT provide full code solutions.
- Do NOT hallucinate library parameters. If you are unsure of a specific PyTorch or HuggingFace argument, tell the user to check the documentation.
- Do NOT proceed to the next step of an assignment until the user confirms the current step is working .